In [ ]:
from discovery_utils.utils import search
from discovery_utils.getters import gtr

from discovery_utils import PROJECT_DIR
VECTOR_DB_DIR = PROJECT_DIR / 'tmp/vector_db'

GTR = gtr.GtrGetter(vector_db_path=VECTOR_DB_DIR)

In [ ]:
Search = search.SearchDataset(GTR, GTR.projects_enriched, "config.yaml")

In [ ]:
search_df = Search.do_search()

In [87]:
from discovery_utils.utils import (
    analysis,
    analysis_gtr,
)
import importlib
importlib.reload(analysis);
importlib.reload(analysis_gtr);

In [ ]:
df = (
    search_df
    .merge(GTR.get_projects_text(), on='id', how='left')
    .query("_score_avg > 0.3")
)
print(len(df))
df_dedup = analysis_gtr.deduplicate_projects(df, description_column='text')
print(len(df_dedup))

In [ ]:
analysis_gtr.funding_per_period(df, period='year', min_year=2010, max_year=2024)

In [ ]:
ts_df = analysis_gtr.get_timeseries(df, period='year', min_year=2010, max_year=2024)
analysis.magnitude_growth(ts_df, 2019, 2024)
# ts_df

In [ ]:
from discovery_utils.getters import crunchbase
CB = crunchbase.CrunchbaseGetter(vector_db_path=VECTOR_DB_DIR)

In [ ]:
SearchCB = search.SearchDataset(CB, CB.organisations_enriched, "config.yaml")
search_cb_df = SearchCB.do_search()

In [ ]:
orgs_df = CB.organisations_enriched.copy()
funds_df = CB.funding_rounds_enriched.copy()

In [116]:
df = (
    search_cb_df
    .query("_score_avg > 0.3")
)

In [204]:
from discovery_utils.utils import (
    analysis,
    analysis_crunchbase,
)
importlib.reload(analysis);
importlib.reload(analysis_crunchbase);

In [ ]:
analysis_crunchbase.orgs_founded_per_period(df, 'year', 2010, 2024)

In [133]:
matching_ids = df.id.to_list()

In [134]:
selected_funding_df = (
    CB.funding_rounds_enriched
    .query("org_id in @matching_ids")
    .query(f"year >= {2010}")
    .query(f"year <= {2024}")
    # .query(f"investment_type in @include_deals")
    .drop_duplicates("funding_round_id")
)

In [191]:
importlib.reload(analysis_crunchbase);

In [ ]:
analysis_crunchbase

In [ ]:
deals_df, deal_counts_df = analysis_crunchbase.get_funding_by_year_and_range(selected_funding_df, 2014, 2024)
deals_df

In [ ]:
deal_counts_df

In [ ]:
(
    selected_funding_df
    .query("announced_on >= '2015-01-01'")
    .query("announced_on < '2016-01-01'")
    .raised_amount_gbp.sum()
)

In [276]:
importlib.reload(analysis_crunchbase);
ts_df = analysis_crunchbase.get_timeseries(df, selected_funding_df, 'year', 2010, 2024)

In [ ]:
ts_df

In [ ]:
analysis.magnitude_growth(ts_df, 2019, 2024)

In [285]:
importlib.reload(analysis_crunchbase);

In [ ]:
aggregated_funding_types_df = analysis_crunchbase.aggregate_by_funding_round_types(selected_funding_df)
analysis_crunchbase.chart_investment_types(aggregated_funding_types_df)

In [ ]:
analysis_crunchbase.chart_deal_sizes(deals_df)

In [ ]:
analysis_crunchbase.chart_deal_sizes_counts(deal_counts_df)